# 🚀 CUDA Multi-Head Attention Benchmark Suite

This notebook allows you to easily:
1. **Auto-detect your Colab GPU** (T4, V100, A100, L4) and configure `nvcc` flags dynamically.
2. **Run any single CUDA kernel** with automatic compilation and error reporting.
3. **Automatically benchmark all Multi-Head Attention kernel versions** in `kernels/multiheadatten/` side-by-side (`1st.cu`, `2nd.cu`, etc.).
4. **Visualize speedup & latency progression** across your kernel improvements.
5. **Compare against PyTorch native FlashAttention / cuDNN** as a ceiling reference.

In [ ]:
# Cell 1: Environment Setup & GPU Architecture Auto-Detection
import os, sys, subprocess, shutil

# 1. Display GPU Info
!nvidia-smi

# 2. Auto-detect GPU Compute Capability for nvcc
ARCH_FLAG = "-arch=sm_75"  # Default fallback for Colab T4
try:
    import torch
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability()
        ARCH_FLAG = f"-arch=sm_{major}{minor}"
        gpu_name = torch.cuda.get_device_name(0)
        print(f"\n[GPU Detected] {gpu_name} (Capability {major}.{minor} -> {ARCH_FLAG})")
    else:
        print("\n[Warning] PyTorch CUDA not found. Defaulting to -arch=sm_75")
except Exception as e:
    print(f"\n[Info] GPU detection notice: {e}. Defaulting to -arch=sm_75")

# 3. Clone or Sync Repository
REPO_URL = "https://github.com/Aakash-Rajput-2024/cuda-training.git"
REPO_DIR = "/content/repo"

if os.path.exists("/content"):
    if not os.path.exists(REPO_DIR):
        print(f"\nCloning repo into {REPO_DIR}...")
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"\nRepo already exists at {REPO_DIR}. Fetching latest updates...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"])
    ROOT_DIR = REPO_DIR
else:
    ROOT_DIR = "."

KERNELS_DIR = os.path.join(ROOT_DIR, "kernels")
MHA_DIR = os.path.join(KERNELS_DIR, "multiheadatten")

print(f"Kernels directory: {KERNELS_DIR}")
print(f"MHA directory:     {MHA_DIR}")


## 🎯 1. Run Any Single Kernel
Specify any `.cu` file relative to `kernels/` (for example `multiheadatten/1st.cu` or `testing.cu`).

In [ ]:
# Change this to any kernel path you want to run
FILE = "multiheadatten/1st.cu"

src_path = os.path.join(KERNELS_DIR, FILE)
safe_name = FILE.replace("/", "_").replace("\\", "_")
exe_path = f"/tmp/{os.path.splitext(safe_name)[0]}"

if not os.path.exists(src_path):
    print(f"[Error] File not found: {src_path}")
    if os.path.exists(KERNELS_DIR):
        print("Available in kernels/:", os.listdir(KERNELS_DIR))
    if os.path.exists(MHA_DIR):
        print("Available in multiheadatten/:", os.listdir(MHA_DIR))
else:
    print(f"Compiling: {src_path} -> {exe_path} using {ARCH_FLAG}...")
    compile_cmd = ["nvcc", "-O2", ARCH_FLAG, "-lineinfo", src_path, "-o", exe_path]
    b = subprocess.run(compile_cmd, capture_output=True, text=True)
    if b.returncode != 0:
        print("\n❌ Compilation Failed:")
        print(b.stderr)
    else:
        print("✓ Compilation Succeeded! Running kernel...\n" + "="*60)
        run_res = subprocess.run([exe_path], capture_output=True, text=True)
        print(run_res.stdout, end="")
        if run_res.stderr:
            print("\n[Stderr]:\n", run_res.stderr)


## 📊 2. Multi-Head Attention Benchmark Suite
This cell automatically scans `kernels/multiheadatten/` for all `.cu` files (e.g. `1st.cu`, `2nd.cu`, etc.), compiles each one, runs them with identical parameters, verifies correctness, and computes **speedup vs the baseline (`1st.cu`)**.

In [ ]:
import re
import pandas as pd

# Benchmark Parameters
N_VAL = 512        # Sequence length
D_MODEL = 512      # Model dimension
HEADS = 8          # Number of heads

print(f"Running MHA Benchmark Suite: N={N_VAL}, d_model={D_MODEL}, heads={HEADS}...")

if not os.path.exists(MHA_DIR):
    print(f"[Error] MHA directory not found at: {MHA_DIR}")
    mha_files = []
else:
    mha_files = sorted([f for f in os.listdir(MHA_DIR) if f.endswith(".cu")])

results = []

def parse_kernel_output(output):
    passed = "PASS" in output
    failed = "FAIL" in output
    status = "PASS" if (passed and not failed) else ("FAIL" if failed else "UNKNOWN")
    
    lat_m = re.search(r"Avg Latency\s*:\s*([\d\.]+)\s*ms", output)
    gflop_m = re.search(r"Throughput\s*:\s*([\d\.]+)\s*GFLOP/s", output)
    
    latency_ms = float(lat_m.group(1)) if lat_m else None
    gflops = float(gflop_m.group(1)) if gflop_m else None
    return status, latency_ms, gflops

baseline_latency = None

for fname in mha_files:
    src_file = os.path.join(MHA_DIR, fname)
    exe_file = f"/tmp/mha_{os.path.splitext(fname)[0]}"
    
    print(f"\n--> Compiling & Running: {fname}...")
    build = subprocess.run(["nvcc", "-O2", ARCH_FLAG, "-lineinfo", src_file, "-o", exe_file],
                           capture_output=True, text=True)
    if build.returncode != 0:
        print(f"    ❌ Compile error in {fname}:\n{build.stderr[:400]}")
        results.append({
            "Kernel": fname,
            "Status": "COMPILE ERROR",
            "Latency (ms)": None,
            "Throughput (GFLOP/s)": None,
            "Speedup vs 1st": None
        })
        continue
    
    run_p = subprocess.run([exe_file, str(N_VAL), str(D_MODEL), str(HEADS)],
                           capture_output=True, text=True)
    status, lat_ms, gflops = parse_kernel_output(run_p.stdout)
    
    if "1st" in fname and lat_ms and baseline_latency is None:
        baseline_latency = lat_ms
    elif baseline_latency is None and lat_ms:
        baseline_latency = lat_ms
        
    speedup = (baseline_latency / lat_ms) if (baseline_latency and lat_ms) else None
    
    results.append({
        "Kernel": fname,
        "Status": status,
        "Latency (ms)": round(lat_ms, 4) if lat_ms else None,
        "Throughput (GFLOP/s)": round(gflops, 2) if gflops else None,
        "Speedup vs 1st": f"{speedup:.2f}x" if speedup else "-"
    })

df_results = pd.DataFrame(results)
print("\n" + "="*65)
print("           MULTI-HEAD ATTENTION BENCHMARK RESULTS")
print("="*65)
try:
    from IPython.display import display
    display(df_results)
except Exception:
    print(df_results.to_string(index=False))


## 📈 3. Visual Progression Plot
Visual comparison showing how latency drops and speedup grows with each optimization.

In [ ]:
import matplotlib.pyplot as plt

valid_df = df_results[df_results["Latency (ms)"].notnull()].copy()

if not valid_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Latency Bar Chart
    bars1 = axes[0].bar(valid_df["Kernel"], valid_df["Latency (ms)"], color="#3b82f6", edgecolor="#1d4ed8", width=0.5)
    axes[0].set_title(f"Latency (ms) [Lower is Better]\n(N={N_VAL}, d_model={D_MODEL}, h={HEADS})", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Latency (ms)")
    axes[0].grid(axis="y", linestyle="--", alpha=0.6)
    for bar in bars1:
        yval = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2.0, yval, f"{yval:.3f} ms", ha="center", va="bottom", fontsize=9, fontweight="bold")

    # 2. Speedup Bar Chart
    num_speedup = []
    for s in valid_df["Speedup vs 1st"]:
        try:
            num_speedup.append(float(str(s).replace("x", "")))
        except:
            num_speedup.append(1.0)
    valid_df["Speedup_Num"] = num_speedup
    
    bars2 = axes[1].bar(valid_df["Kernel"], valid_df["Speedup_Num"], color="#10b981", edgecolor="#047857", width=0.5)
    axes[1].axhline(1.0, color="gray", linestyle="--", label="1.0x Baseline (1st.cu)")
    axes[1].set_title("Speedup vs Baseline (1st.cu) [Higher is Better]", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Speedup Multiplier")
    axes[1].grid(axis="y", linestyle="--", alpha=0.6)
    axes[1].legend()
    for bar in bars2:
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2.0, yval, f"{yval:.2f}x", ha="center", va="bottom", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("No successful benchmark runs to plot yet.")


## ⚡ 4. PyTorch Scaled Dot-Product Attention (Reference Ceiling)
Measures PyTorch native `F.scaled_dot_product_attention` (FlashAttention / cuDNN) on the same GPU for the exact same $(N, d_{model}, h)$ inputs to serve as the reference ceiling.

In [ ]:
import torch
import torch.nn.functional as F

if torch.cuda.is_available():
    dev = torch.device("cuda")
    dk = D_MODEL // HEADS
    
    # Tensor shape: (batch_size=1, heads=h, seq_len=N, head_dim=dk)
    q = torch.randn(1, HEADS, N_VAL, dk, device=dev, dtype=torch.float32)
    k = torch.randn(1, HEADS, N_VAL, dk, device=dev, dtype=torch.float32)
    v = torch.randn(1, HEADS, N_VAL, dk, device=dev, dtype=torch.float32)
    
    # Warmup
    for _ in range(10):
        _ = F.scaled_dot_product_attention(q, k, v)
    torch.cuda.synchronize()
    
    # Benchmark with CUDA events
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev = torch.cuda.Event(enable_timing=True)
    
    iters = 50
    start_ev.record()
    for _ in range(iters):
        _ = F.scaled_dot_product_attention(q, k, v)
    end_ev.record()
    torch.cuda.synchronize()
    
    pyt_ms = start_ev.elapsed_time(end_ev) / iters
    flops = 4.0 * N_VAL * N_VAL * D_MODEL
    pyt_gflops = flops / (pyt_ms * 1e6)
    
    print("===================================================================")
    print(" PyTorch Native F.scaled_dot_product_attention (cuDNN / Flash)")
    print("===================================================================")
    print(f"  Config:      N={N_VAL}, d_model={D_MODEL}, heads={HEADS}, dk={dk}")
    print(f"  Avg Latency: {pyt_ms:.4f} ms")
    print(f"  Throughput:  {pyt_gflops:.2f} GFLOP/s")
    if baseline_latency:
        print(f"  Gap:         Baseline 1st.cu is {baseline_latency / pyt_ms:.1f}x slower than PyTorch native")
    print("===================================================================")
else:
    print("PyTorch CUDA is not available in this environment.")
